In [21]:
import pandas as pd

train_data_encoded = pd.read_csv('../data/train_encoded.csv')
test_data_encoded = pd.read_csv('../data/test_encoded.csv')

# GBA + GAM

In [ ]:
import os
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from pygam import LinearGAM, s
from xgboost import XGBRegressor
from tqdm.auto import tqdm

# ---------------------------------------
# 2. Load Training Data
# ---------------------------------------
X = train_data_encoded.drop('y', axis=1).values  # Make sure X is a numpy array
y = train_data_encoded['y'].values
n_features = X.shape[1]

# ---------------------------------------
# 3. Set Best Hyperparameters for Base Models
# ---------------------------------------
# Fill these from your best CV results
best_lam = 100          # Best lambda for GAM
best_n_splines = 10    # Best number of splines for GAM
best_est = 1000          # Best n_estimators for XGB

best_est = 500         # Best number of estimater for XGB
best_depth = 6        # Best max_depth for XGB
best_learn = 0.1        # Best learning_rate for XGB
best_sub = 0.7          # Best subsample for XGB
best_bytree = 1       # Best colsample_bytree for XGB

# ---------------------------------------
# 4. Cross-Validation to Train Stacking Meta-Model
# ---------------------------------------
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Create containers for Out-of-Fold (OOF) predictions
oof_preds_gam = np.zeros_like(y, dtype=float)
oof_preds_xgb = np.zeros_like(y, dtype=float)

# Build OOF predictions by training base models inside each fold
for train_idx, val_idx in tqdm(kf.split(X,y), total=5, desc="Cross-Validation"):
    X_tr, X_val = X[train_idx], X[val_idx]   
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train GAM on capped data
    terms = s(0, n_splines=best_n_splines)
    for i in range(1, n_features):
        terms += s(i, n_splines=best_n_splines)
    gam = LinearGAM(terms, lam=best_lam)
    gam.fit(X_tr, y_tr)
    oof_preds_gam[val_idx] = gam.predict(X_val)

    # Train XGBoost on raw data
    gbm = XGBRegressor(
        n_estimators=best_est,
        max_depth=best_depth,
        learning_rate=best_learn,
        subsample=best_sub,
        colsample_bytree=best_bytree,
        random_state=42,
        n_jobs=-1
    )
    gbm.fit(X_tr, y_tr)
    oof_preds_xgb[val_idx] = gbm.predict(X_val)


# ---------------------------------------
# 5. Train Meta-Model (Ridge Regression)
# ---------------------------------------
# Stack OOF predictions as features
stacked_X = np.vstack([oof_preds_gam, oof_preds_xgb]).T

# Train Ridge Regression to learn stacking weights
meta_model = Ridge(alpha=1.0)
meta_model.fit(stacked_X, y)

# Print stacking weights
print("\nLearned stacking weights:")
print(f"GAM weight: {meta_model.coef_[0]:.4f}")
print(f"XGBoost weight: {meta_model.coef_[1]:.4f}")

# ---------------------------------------
# 6. Evaluate Stacking Performance on OOF Predictions
# ---------------------------------------
y_oof_pred_stacked = meta_model.predict(stacked_X)

mse = mean_squared_error(y, y_oof_pred_stacked)
r2 = r2_score(y, y_oof_pred_stacked)

print(f"\n5-Fold CV Stacked Model Performance:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.4f}")

Cross-Validation: 100%|██████████| 5/5 [01:17<00:00, 15.41s/it]


Learned stacking weights:
GAM weight: 0.3560
XGBoost weight: 0.6474

5-Fold CV Stacked Model Performance:
Mean Squared Error (MSE): 4554.48
R² Score: 0.7821


In [ ]:
# ---------------------------------------
# 7. Retrain Base Models on Full Data
# ---------------------------------------
# Retrain GAM on full (X, y)
terms = s(0, n_splines=best_n_splines)
for i in range(1, n_features):
    terms += s(i, n_splines=best_n_splines)
gam_full = LinearGAM(terms, lam=best_lam)
gam_full.fit(X, y)

# Retrain XGBoost on full (X, y)
gbm_full = XGBRegressor(
    n_estimators=best_est,
    max_depth=best_depth,
    learning_rate=best_learn,
    subsample=best_sub,
    colsample_bytree=best_bytree,
    random_state=42,
    n_jobs=-1
)
gbm_full.fit(X, y)

# ---------------------------------------
# 8. Predict on Test Set
# ---------------------------------------
# Load test data
X_test = pd.read_csv('../data/test_encoded.csv').values

# Predict using full models
y_test_gam = gam_full.predict(X_test)
y_test_xgb = gbm_full.predict(X_test)

# Stack test predictions
stacked_X_test = np.vstack([y_test_gam, y_test_xgb]).T
y_test_stacked = meta_model.predict(stacked_X_test)

# Optionally, clip negative predictions
y_test_stacked = np.clip(y_test_stacked, 0, None)

# ---------------------------------------
# 9. Save Final Stacked Predictions
# ---------------------------------------
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_GBM_GAM.txt', y_test_stacked, fmt='%.6f')

print("\nFinal stacked predictions saved to '../prediction/predicted_GBM_GAM.txt'.") # 38.44 (GBM + GAM)


Final stacked predictions saved to '../prediction/predicted_Stacking_Clean.txt'.


# GBM + GAM + Elastic Net

In [28]:
# Process ENET data
# find numeriacal value
X_ENET = train_data_encoded.drop('y',axis=1)
X_ENET_test = test_data_encoded

numerical_features = []
for f in X_ENET.columns:
    if X_ENET[f].dtype in ['int64','float64']:
        numerical_features.append(f)

# cap outliers
for f in numerical_features:
    lower = X_ENET[f].quantile(0.01)
    upper = X_ENET[f].quantile(0.99)
    X_ENET[f] = X_ENET[f].clip(lower, upper)
    
    lower = X_ENET_test[f].quantile(0.01)
    upper = X_ENET_test[f].quantile(0.99)
    X_ENET_test[f] = X_ENET_test[f].clip(lower, upper)
print('Outliers have been capped at 1st and 99th percentiles')

# scale numerical columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_ENET[numerical_features] = scaler.fit_transform(X_ENET[numerical_features])
X_ENET_test[numerical_features] = scaler.fit_transform(X_ENET_test[numerical_features])
print('Columns are scaled')

Outliers have been capped at 1st and 99th percentiles
Columns are scaled


In [29]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNetCV
from pygam import LinearGAM, s
from xgboost import XGBRegressor
from tqdm.auto import tqdm

# ---------------------------------------
# 2. Load Training Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values
X_ENET_values = X_ENET.values
n_features = X.shape[1]

# ---------------------------------------
# 3. Set Best Hyperparameters for Base Models
# ---------------------------------------
best_lam = 100        # Best lambda for GAM
best_n_splines = 10   # Best number of splines for GAM

best_est = 500        # Best n_estimators for XGB
best_depth = 6        # Best max_depth for XGB
best_learn = 0.1      # Best learning_rate for XGB
best_sub = 0.7        # Best subsample for XGB
best_bytree = 1       # Best colsample_bytree for XGB

# ---------------------------------------
# 4. Cross-Validation to Train Stacking Meta-Model
# ---------------------------------------
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Create containers for Out-of-Fold (OOF) predictions
oof_preds_gam = np.zeros_like(y, dtype=float)
oof_preds_xgb = np.zeros_like(y, dtype=float)
oof_preds_enet = np.zeros_like(y, dtype=float)

for train_idx, val_idx in tqdm(kf.split(X,y), total=5, desc="Cross-Validation"):
    X_tr, X_val = X[train_idx], X[val_idx]
    X_tr_ENET, X_val_ENET = X_ENET_values[train_idx], X_ENET_values[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train GAM
    terms = s(0, n_splines=best_n_splines)
    for i in range(1, n_features):
        terms += s(i, n_splines=best_n_splines)
    gam = LinearGAM(terms, lam=best_lam)
    gam.fit(X_tr, y_tr)
    oof_preds_gam[val_idx] = gam.predict(X_val)

    # Train XGBoost
    gbm = XGBRegressor(
        n_estimators=best_est,
        max_depth=best_depth,
        learning_rate=best_learn,
        subsample=best_sub,
        colsample_bytree=best_bytree,
        random_state=42,
        n_jobs=-1
    )
    gbm.fit(X_tr, y_tr)
    oof_preds_xgb[val_idx] = gbm.predict(X_val)

    # Train Elastic Net (with internal CV to find best alpha/l1_ratio)
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0], 
        alphas=np.logspace(-4, 2, 50), 
        cv=5, 
        random_state=42,
        max_iter=5000
    )
    enet.fit(X_tr_ENET, y_tr)
    oof_preds_enet[val_idx] = enet.predict(X_val_ENET)

# ---------------------------------------
# 5. Train Meta-Model (Ridge Regression)
# ---------------------------------------
# Stack OOF predictions as features
stacked_X = np.vstack([oof_preds_gam, oof_preds_xgb, oof_preds_enet]).T

# Train Ridge Regression to learn stacking weights
meta_model = Ridge(alpha=1.0)
meta_model.fit(stacked_X, y)

# Print stacking weights
print("\nLearned stacking weights:")
print(f"GAM weight: {meta_model.coef_[0]:.4f}")
print(f"XGBoost weight: {meta_model.coef_[1]:.4f}")
print(f"ElasticNet weight: {meta_model.coef_[2]:.4f}")

# ---------------------------------------
# 6. Evaluate Stacking Performance on OOF Predictions
# ---------------------------------------
y_oof_pred_stacked = meta_model.predict(stacked_X)

mse = mean_squared_error(y, y_oof_pred_stacked)
r2 = r2_score(y, y_oof_pred_stacked)

print(f"\n5-Fold CV Stacked Model Performance:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.4f}")


Cross-Validation: 100%|██████████| 5/5 [01:30<00:00, 18.14s/it]


Learned stacking weights:
GAM weight: 0.3549
XGBoost weight: 0.6472
ElasticNet weight: 0.0014

5-Fold CV Stacked Model Performance:
Mean Squared Error (MSE): 4554.48
R² Score: 0.7821


In [ ]:
# ---------------------------------------
# 7. Retrain Base Models on Full Data
# ---------------------------------------
# Retrain GAM
terms = s(0, n_splines=best_n_splines)
for i in range(1, n_features):
    terms += s(i, n_splines=best_n_splines)
gam_full = LinearGAM(terms, lam=best_lam)
gam_full.fit(X, y)

# Retrain XGBoost
gbm_full = XGBRegressor(
    n_estimators=best_est,
    max_depth=best_depth,
    learning_rate=best_learn,
    subsample=best_sub,
    colsample_bytree=best_bytree,
    random_state=42,
    n_jobs=-1
)
gbm_full.fit(X, y)

# Retrain Elastic Net
enet_full = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.9, 1.0], 
    alphas=np.logspace(-4, 2, 50), 
    cv=5, 
    random_state=42,
    max_iter=5000
)
enet_full.fit(X_ENET_values, y)

# ---------------------------------------
# 8. Predict on Test Set
# ---------------------------------------
X_test = pd.read_csv('../data/test_encoded.csv').values
X_ENET_test_values = X_ENET_test.values

# Predict with full models
y_test_gam = gam_full.predict(X_test)
y_test_xgb = gbm_full.predict(X_test)
y_test_enet = enet_full.predict(X_ENET_test_values)

# Stack test predictions
stacked_X_test = np.vstack([y_test_gam, y_test_xgb, y_test_enet]).T
y_test_stacked = meta_model.predict(stacked_X_test)

# Optionally clip negatives
y_test_stacked = np.clip(y_test_stacked, 0, None)

# ---------------------------------------
# 9. Save Final Stacked Predictions
# ---------------------------------------
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_GBM_GAM_ENET.txt', y_test_stacked, fmt='%.6f')

print("\nFinal stacked predictions saved to '../prediction/predicted_GBM_GAM_ENET.txt'.") # 38.42


Final stacked predictions saved to '../prediction/predicted_GAM_XGB_ENET.txt'.


# GBM + ENET 

In [31]:
# Process ENET data
# find numeriacal value
X_ENET = train_data_encoded.drop('y',axis=1)
X_ENET_test = test_data_encoded

numerical_features = []
for f in X_ENET.columns:
    if X_ENET[f].dtype in ['int64','float64']:
        numerical_features.append(f)

# cap outliers
for f in numerical_features:
    lower = X_ENET[f].quantile(0.01)
    upper = X_ENET[f].quantile(0.99)
    X_ENET[f] = X_ENET[f].clip(lower, upper)
    
    lower = X_ENET_test[f].quantile(0.01)
    upper = X_ENET_test[f].quantile(0.99)
    X_ENET_test[f] = X_ENET_test[f].clip(lower, upper)
print('Outliers have been capped at 1st and 99th percentiles')

# scale numerical columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_ENET[numerical_features] = scaler.fit_transform(X_ENET[numerical_features])
X_ENET_test[numerical_features] = scaler.fit_transform(X_ENET_test[numerical_features])
print('Columns are scaled')

Outliers have been capped at 1st and 99th percentiles
Columns are scaled


In [32]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNetCV
from pygam import LinearGAM, s
from xgboost import XGBRegressor
from tqdm.auto import tqdm

# ---------------------------------------
# 2. Load Training Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values
X_ENET_values = X_ENET.values
n_features = X.shape[1]

# ---------------------------------------
# 3. Set Best Hyperparameters for Base Models
# ---------------------------------------
best_est = 500        # Best n_estimators for XGB
best_depth = 6        # Best max_depth for XGB
best_learn = 0.1      # Best learning_rate for XGB
best_sub = 0.7        # Best subsample for XGB
best_bytree = 1       # Best colsample_bytree for XGB

# ---------------------------------------
# 4. Cross-Validation to Train Stacking Meta-Model
# ---------------------------------------
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Create containers for Out-of-Fold (OOF) predictions
oof_preds_xgb = np.zeros_like(y, dtype=float)
oof_preds_enet = np.zeros_like(y, dtype=float)

for train_idx, val_idx in tqdm(kf.split(X,y), total=5, desc="Cross-Validation"):
    X_tr, X_val = X[train_idx], X[val_idx]
    X_tr_ENET, X_val_ENET = X_ENET_values[train_idx], X_ENET_values[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train XGBoost
    gbm = XGBRegressor(
        n_estimators=best_est,
        max_depth=best_depth,
        learning_rate=best_learn,
        subsample=best_sub,
        colsample_bytree=best_bytree,
        random_state=42,
        n_jobs=-1
    )
    gbm.fit(X_tr, y_tr)
    oof_preds_xgb[val_idx] = gbm.predict(X_val)

    # Train Elastic Net (with internal CV to find best alpha/l1_ratio)
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0], 
        alphas=np.logspace(-4, 2, 50), 
        cv=5, 
        random_state=42,
        max_iter=5000
    )
    enet.fit(X_tr_ENET, y_tr)
    oof_preds_enet[val_idx] = enet.predict(X_val_ENET)

# ---------------------------------------
# 5. Train Meta-Model (Ridge Regression)
# ---------------------------------------
# Stack OOF predictions as features
stacked_X = np.vstack([oof_preds_xgb, oof_preds_enet]).T

# Train Ridge Regression to learn stacking weights
meta_model = Ridge(alpha=1.0)
meta_model.fit(stacked_X, y)

# Print stacking weights
print("\nLearned stacking weights:")
print(f"XGBoost weight: {meta_model.coef_[0]:.4f}")
print(f"ElasticNet weight: {meta_model.coef_[1]:.4f}")

# ---------------------------------------
# 6. Evaluate Stacking Performance on OOF Predictions
# ---------------------------------------
y_oof_pred_stacked = meta_model.predict(stacked_X)

mse = mean_squared_error(y, y_oof_pred_stacked)
r2 = r2_score(y, y_oof_pred_stacked)

print(f"\n5-Fold CV Stacked Model Performance:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.4f}")


Cross-Validation: 100%|██████████| 5/5 [00:43<00:00,  8.72s/it]


Learned stacking weights:
XGBoost weight: 0.7267
ElasticNet weight: 0.2799

5-Fold CV Stacked Model Performance:
Mean Squared Error (MSE): 4642.83
R² Score: 0.7779


In [ ]:
# ---------------------------------------
# 7. Retrain Base Models on Full Data
# ---------------------------------------
# Retrain GAM
terms = s(0, n_splines=best_n_splines)
for i in range(1, n_features):
    terms += s(i, n_splines=best_n_splines)
gam_full = LinearGAM(terms, lam=best_lam)
gam_full.fit(X, y)

# Retrain XGBoost
gbm_full = XGBRegressor(
    n_estimators=best_est,
    max_depth=best_depth,
    learning_rate=best_learn,
    subsample=best_sub,
    colsample_bytree=best_bytree,
    random_state=42,
    n_jobs=-1
)
gbm_full.fit(X, y)

# Retrain Elastic Net
enet_full = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.9, 1.0], 
    alphas=np.logspace(-4, 2, 50), 
    cv=5, 
    random_state=42,
    max_iter=5000
)
enet_full.fit(X_ENET_values, y)

# ---------------------------------------
# 8. Predict on Test Set
# ---------------------------------------
X_test = pd.read_csv('../data/test_encoded.csv').values
X_ENET_test_values = X_ENET_test.values

# Predict with full models
y_test_xgb = gbm_full.predict(X_test)
y_test_enet = enet_full.predict(X_ENET_test_values)

# Stack test predictions
stacked_X_test = np.vstack([y_test_xgb, y_test_enet]).T
y_test_stacked = meta_model.predict(stacked_X_test)

# Optionally clip negatives
y_test_stacked = np.clip(y_test_stacked, 0, None)

# ---------------------------------------
# 9. Save Final Stacked Predictions
# ---------------------------------------
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_GBM_ENET.txt', y_test_stacked, fmt='%.6f')

print("\nFinal stacked predictions saved to '../prediction/predicted_GBM_ENET.txt'.") # 38.09


Final stacked predictions saved to '../prediction/predicted_GBM_ENET.txt'.
